# Lab 03 · Tools and MCP

**Prepared by Mustafa Assaf** · Beginner level · 60–90 minute session

Start with the small worked example, read the expected output, then change one thing.
Use a CPU runtime and run the cells from top to bottom. No repository clone is needed.

**Practice mode:** model answers are explicitly scripted so you can inspect the code
without a key. They demonstrate control flow, not model quality. MCP, A2A and LangGraph
examples run their actual libraries locally even in practice mode; their setup cells
install pinned packages if needed.

**Optional live calls:** add `FREEINFERENCE_API_KEY` in Colab's Secrets panel, allow
this notebook access, and change `MODE` to `"live"`. The initial model is
`qwen3.6-35b` on FreeInference; `BASE_URL`, `MODEL` and `SECRET_NAME` can be changed
for another compatible provider. Keep the key out of cells and outputs.

### Shared setup
The next cell contains the reusable model client. Run it as supplied; the short teaching examples follow below.

In [ ]:
"""Small, provider-configurable teaching client. Standard library only."""
import getpass
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request

# Shared by every Client in this notebook/process, including parallel workflows.
_NETWORK_SLOTS = threading.BoundedSemaphore(2)


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None  # Never forward a credential to another host.


class Client:
    """Practice responses are explicit fixtures, not an LLM or model simulation."""
    def __init__(self, mode="practice", model="", family="", base_url="", api_key="",
                 max_calls=24, max_tokens=512, timeout=45, token_field="max_tokens", extra_body=None):
        if extra_body is not None and (not isinstance(extra_body, dict) or
                set(extra_body) & {"model", "messages", "max_tokens", "max_completion_tokens", "stream"}):
            raise ValueError("extra_body must be a dict and cannot override the request contract or budgets")
        if mode not in {"practice", "live"}:
            raise ValueError("mode must be practice or live")
        if token_field not in {"max_tokens", "max_completion_tokens"}:
            raise ValueError("Check which output-token field your provider supports")
        if max_calls < 1 or max_tokens < 1 or timeout <= 0:
            raise ValueError("Call, output, and timeout bounds must be positive")
        if mode == "live":
            url = urllib.parse.urlsplit(base_url)
            if url.scheme != "https" or not url.netloc or url.username or url.password or url.query or url.fragment:
                raise ValueError("Use an HTTPS API base URL, without credentials/query/fragment")
            if not model.strip() or not api_key.strip():
                raise ValueError("Live mode requires a model and secret key")
        self.mode, self.model, self.family = mode, model, family
        self.base_url, self._api_key = base_url.rstrip("/"), api_key
        self.max_calls, self.max_tokens, self.timeout = max_calls, max_tokens, timeout
        self.token_field, self.trace = token_field, []
        self.extra_body = dict(extra_body or {})
        self._freeinference = urllib.parse.urlsplit(self.base_url).hostname == "freeinference.org"
        self._attempts, self._lock = 0, threading.Lock()

    def ask(self, prompt, *, system="You are a concise teaching assistant.", practice=""):
        if not isinstance(prompt, str) or len(prompt) + len(system) > 28000:
            raise ValueError("Prompt must be text and total prompt length <= 28,000 characters")
        with self._lock:
            if self._attempts >= self.max_calls:
                raise RuntimeError("Call budget reached; inspect the trace before increasing it")
            self._attempts += 1
            call_id = self._attempts
        started = time.perf_counter()
        row = dict(call=call_id, mode=self.mode, model=self.model or "scripted-fixture",
                   family=self.family or ("practice" if self.mode == "practice" else None), input_tokens=None, output_tokens=None,
                   cost=None, status="ok")
        try:
            if self.mode == "practice":
                content = practice
            else:
                messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
                body = {"model": self.model, "messages": messages, self.token_field: self.max_tokens}
                body.update(self.extra_body)
                if self._freeinference:
                    body.update({"stream": False, "thinking": {"type": "disabled"}})
                req = urllib.request.Request(self.base_url + "/chat/completions",
                    data=json.dumps(body).encode(), method="POST",
                    headers={"Authorization": "Bearer " + self._api_key, "Content-Type": "application/json"})
                try:
                    with _NETWORK_SLOTS:
                        with urllib.request.build_opener(_NoRedirect).open(req, timeout=self.timeout) as response:
                            payload = json.load(response)
                except urllib.error.HTTPError as exc:
                    if exc.code == 429:
                        raise RuntimeError("Gateway HTTP 429: rate/quota limit. Check your quota; no retry or key rotation made.") from None
                    raise RuntimeError(f"Gateway HTTP {exc.code}; check provider access/configuration. No retry made.") from None
                except (urllib.error.URLError, TimeoutError, OSError, ValueError):
                    raise RuntimeError("Gateway/network/JSON error; inspect provider configuration. No retry made.") from None
                try:
                    choice = payload["choices"][0]
                    if choice.get("finish_reason") == "length":
                        raise RuntimeError("Output truncated; inspect token budget before retrying")
                    content = choice["message"]["content"]
                    if not isinstance(content, str):
                        raise ValueError("Missing text response")
                    usage = payload.get("usage") or {}
                    row["input_tokens"] = usage.get("prompt_tokens")
                    row["output_tokens"] = usage.get("completion_tokens")
                    if self._freeinference and "model is starting up" in content.lower():
                        raise RuntimeError("FreeInference model is starting up. Wait about 120 seconds, then retry manually; this attempt used the call budget.")
                except (KeyError, IndexError, TypeError, ValueError, AttributeError):
                    raise RuntimeError("Gateway response has no usable text; schema/capability check needed") from None
            return content
        except Exception:
            row["status"] = "error"
            raise
        finally:
            row["latency_s"] = round(time.perf_counter() - started, 4)
            with self._lock:
                self.trace.append(row)


def secret(name="FREEINFERENCE_API_KEY"):
    """Read a Colab secret, then fall back to a hidden local prompt. Never print it."""
    try:
        from google.colab import userdata
        key = userdata.get(name)
        if key:
            return key
    except Exception:
        pass
    return getpass.getpass(f"{name} (hidden): ")


def parse_object(text):
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("Expected one JSON object")
    return value


def summarize_trace(clients):
    rows = [row for client in clients for row in client.trace]
    return dict(calls=len(rows), failed_calls=sum(r["status"] != "ok" for r in rows),
                input_tokens=sum(r["input_tokens"] for r in rows) if rows and all(isinstance(r["input_tokens"], int) for r in rows) else None,
                output_tokens=sum(r["output_tokens"] for r in rows) if rows and all(isinstance(r["output_tokens"], int) for r in rows) else None,
                summed_call_latency_s=round(sum(r["latency_s"] for r in rows), 4), cost=None)

In [ ]:
# Run all examples without a key first. Change only MODE for a live model run.
MODE = "practice"  # "live" uses the API key supplied through Colab Secrets
BASE_URL = "https://freeinference.org/v1"
MODEL = "qwen3.6-35b"
SECRET_NAME = "FREEINFERENCE_API_KEY"

client = Client(mode=MODE, base_url=BASE_URL, model=MODEL,
                api_key=secret(SECRET_NAME) if MODE == "live" else "",
                max_calls=24, max_tokens=512)
print("Mode:", MODE)

## What you will learn

Expose one Python function through **Model Context Protocol (MCP)**, discover its input schema, and call it from a separate client. Then check one invalid call. Allow 60–90 minutes, including the exercise.

The example is deliberately small: a study-note lookup. A model can suggest the topic, but Python and the MCP server control execution. The server is a real subprocess speaking MCP over **stdio**. No public server, tunnel, or API key is needed for the protocol demonstration.

## 1. Install the official SDK

Run the collapsed setup cell. It installs the pinned `mcp==2.3.0` package if necessary; the first Colab run needs internet access for this installation. The actual tool calls stay local. Python 3.11 or later is required.

In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    sdk_version = importlib.metadata.version("mcp")
except importlib.metadata.PackageNotFoundError:
    sdk_version = None
if sdk_version != "2.3.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mcp==2.3.0"])

import tempfile
from pathlib import Path
from mcp import Client as MCPClient, StdioServerParameters

## 2. Write one tool

This is the server's useful code. `@server.tool()` exposes the function, its docstring, and the `topic: str` argument as an MCP tool. The SDK builds the input schema.

The function only reads two fixed notes. It does not execute arbitrary model text.

In [ ]:
tool_code = '''
@server.tool()
def lookup_note(topic: str) -> str:
    """Look up one short study note by topic."""
    notes = {
        "prompt": "A prompt is the input sent to a model.",
        "tool": "A tool is a function the host can execute.",
    }
    return notes.get(topic, "No note found.")
'''

The next collapsed cell wraps that function in an MCP server and saves it to a temporary file. The client will launch it with the same Python interpreter. Server diagnostics belong on stderr: stdout is reserved for protocol messages.

In [ ]:
server_folder = tempfile.TemporaryDirectory(prefix="mcp-study-notes-")
server_path = Path(server_folder.name) / "notes_server.py"
server_path.write_text(
    'from mcp.server import MCPServer\n'
    'server = MCPServer("Study notes")\n'
    + tool_code
    + '\nserver.run()\n',
    encoding="utf-8",
)
server_parameters = StdioServerParameters(
    command=sys.executable, args=[str(server_path)]
)

## 3. Suggest an argument

The course client uses a visible fixture in practice mode and the configured provider in live mode. This is a tiny teaching decision, rather than native provider function calling. We validate the allowed argument before any MCP request.

In [ ]:
topic = client.ask(
    "Choose a topic for this question: What is a prompt?",
    system="Return exactly one allowed topic: prompt or tool. No other text.",
    practice="prompt",
).strip()
if topic not in {"prompt", "tool"}:
    raise ValueError("The model did not return an allowed topic.")
print("Selected topic:", topic)

## 4. Discover and call the tool

Entering `async with` connects and negotiates the protocol. `list_tools` returns the discovered schema; `call_tool` sends a real MCP request. Leaving the block closes the session and subprocess.

Expected practice output includes `lookup_note`, `A prompt is the input sent to a model.`, and `Missing argument rejected: True`. The negotiated protocol version is also printed and may vary with compatible SDK releases.

In [ ]:
async with MCPClient(server_parameters) as notes:
    print("MCP version:", notes.protocol_version)
    catalog = await notes.list_tools()
    print("Discovered:", catalog.tools[0].name)
    print("Required arguments:", catalog.tools[0].input_schema["required"])
    result = await notes.call_tool("lookup_note", {"topic": topic})
    print(result.content[0].text)
    invalid = await notes.call_tool("lookup_note", {})
    print("Missing argument rejected:", invalid.is_error)

assert not result.is_error
assert invalid.is_error
assert "topic" in catalog.tools[0].input_schema["required"]

## What happened?

The **host** chose an allowed argument, the **MCP client** transported the request, and the **MCP server** validated and ran the function. MCP standardises the connection and tool contract. It does not choose a useful task, make the model correct, or grant permission to execute arbitrary code.

A missing required argument produces a tool error (`is_error=True`). An unknown but well-typed topic returns `No note found.` because that is our function's chosen behavior. Transport errors and domain results are different cases.

## Your turn

Add a `memory` note, allow that topic in the selection check, and run the tool again. Change the missing-argument request to `{"topic": "unknown"}` and explain why its result is different. Rerun the server-file setup after editing the tool.

**Exit ticket:** Which component executes `lookup_note`, and what did MCP add beyond a direct Python function call?

Further reading: [official MCP Python SDK client guide](https://py.sdk.modelcontextprotocol.io/client/) and [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture). The example uses the official SDK; it is a small local teaching example, with authentication and deployment left for a later lesson.

### Finish
Explain the result in your own words. Make the small change suggested above and rerun its check. If you use a live model, save the actual result and trace; a different or failed response is something to investigate.